# Tool-Calling Hallucination Detection: Reproducible Full Run

This notebook reproduces the final Kaggle full experiment for the NLP course project.

It performs:
1. clean clone from GitHub;
2. dependency installation;
3. repository sanity checks;
4. LettuceDetect smoke test;
5. full dataset construction;
6. baseline evaluation;
7. ModernBERT token-classifier training;
8. final evaluation;
9. report table generation;
10. lightweight result archiving.

Expected final dataset size in the current run:
- train: 2790
- validation: 600
- test: 595
- total: 3985

The final reported metrics are saved to:
- `/kaggle/working/results/kaggle_full/all_metrics.csv`
- `/kaggle/working/results/kaggle_full/per_type_metrics.csv`
- `/kaggle/working/results/kaggle_full/baseline_availability.csv`


In [ ]:
!nvidia-smi -L || true
!python --version
!pwd
!ls -lah /kaggle/working


In [ ]:
!cd /kaggle/working && rm -rf LLMs_final_project
!cd /kaggle/working && git clone https://github.com/marritau/LLMs_final_project.git
!cd /kaggle/working/LLMs_final_project && git log -1 --oneline
!cd /kaggle/working/LLMs_final_project && ls -lah


In [ ]:
!cd /kaggle/working/LLMs_final_project && pip install -q -r requirements-kaggle.txt
!cd /kaggle/working/LLMs_final_project && pip install -q -e ".[dev]"


In [ ]:
!cd /kaggle/working/LLMs_final_project && grep -n "lettuce_model_path" scripts/_run_utils.py configs/kaggle_small.yaml configs/kaggle_full.yaml configs/kaggle_full_safe.yaml
!cd /kaggle/working/LLMs_final_project && ls scripts | grep 00_check_lettucedetect
!cd /kaggle/working/LLMs_final_project && python -m py_compile scripts/_run_utils.py scripts/00_check_lettucedetect.py


In [ ]:
# Expected: all tests should pass.
!cd /kaggle/working/LLMs_final_project && python -m pytest -q


In [ ]:
# Expected output should contain a detected span for the trivial contradiction:
# The capital of France is Berlin.
# This confirms that real LettuceDetect works and that model_path is passed correctly.
!cd /kaggle/working/LLMs_final_project && python scripts/00_check_lettucedetect.py


In [ ]:
# Expected output for the current full configuration:
# train: 2790 records
# validation: 600 records
# test: 595 records
!cd /kaggle/working/LLMs_final_project && python scripts/01_build_dataset.py --config configs/kaggle_full.yaml


In [ ]:
!ls -lh /kaggle/working/artifacts/kaggle_full/dataset
!wc -l /kaggle/working/artifacts/kaggle_full/dataset/train.jsonl
!wc -l /kaggle/working/artifacts/kaggle_full/dataset/validation.jsonl
!wc -l /kaggle/working/artifacts/kaggle_full/dataset/test.jsonl


In [ ]:
# This runs trivial, random, keyword, value checker, TF-IDF logistic regression,
# Lookback Lens-style proxy, attention Lookback-style adapted baseline,
# NLI sentence verifier, and real LettuceDetect.
!cd /kaggle/working/LLMs_final_project && python scripts/02_run_baselines.py --config configs/kaggle_full.yaml


In [ ]:
# Expected important line: lettucedetect_real,real,
!cat /kaggle/working/results/kaggle_full/baseline_availability.csv


In [ ]:
!cat /kaggle/working/results/kaggle_full/baseline_metrics.csv


In [ ]:
!rm -rf /kaggle/working/artifacts/kaggle_full/modernbert-token-classifier


In [ ]:
# CUDA_VISIBLE_DEVICES=0 is intentional: Kaggle may provide two T4 GPUs,
# but this project uses one GPU for a more stable and reproducible run.
# Expected final output should include:
# /kaggle/working/artifacts/kaggle_full/modernbert-token-classifier
!cd /kaggle/working/LLMs_final_project && CUDA_VISIBLE_DEVICES=0 python scripts/03_train_modernbert.py --config configs/kaggle_full.yaml


In [ ]:
# Important: --model-path is required so evaluation uses the already trained model instead of retraining.
!cd /kaggle/working/LLMs_final_project && CUDA_VISIBLE_DEVICES=0 python scripts/04_evaluate.py \
  --config configs/kaggle_full.yaml \
  --model-path /kaggle/working/artifacts/kaggle_full/modernbert-token-classifier


In [ ]:
# Expected output: Report tables written to /kaggle/working/results/kaggle_full
!cd /kaggle/working/LLMs_final_project && python scripts/06_make_report_tables.py --config configs/kaggle_full.yaml


In [ ]:
!cat /kaggle/working/results/kaggle_full/baseline_availability.csv
!cat /kaggle/working/results/kaggle_full/all_metrics.csv
!cat /kaggle/working/results/kaggle_full/per_type_metrics.csv
!cat /kaggle/working/results/kaggle_full/dataset_statistics.csv


In [ ]:
import pandas as pd

base = "/kaggle/working/results/kaggle_full"

availability = pd.read_csv(f"{base}/baseline_availability.csv")
all_metrics = pd.read_csv(f"{base}/all_metrics.csv")
per_type = pd.read_csv(f"{base}/per_type_metrics.csv")
dataset_stats = pd.read_csv(f"{base}/dataset_statistics.csv")

display(availability)
display(
    all_metrics[
        [
            "method",
            "sentence_macro_f1",
            "sentence_balanced_accuracy",
            "sentence_f1",
            "span_char_f1",
            "span_relaxed_span_f1",
        ]
    ].sort_values("sentence_macro_f1", ascending=False)
)
display(per_type)
display(dataset_stats)


In [ ]:
summary_cols = [
    "method",
    "sentence_macro_f1",
    "sentence_balanced_accuracy",
    "sentence_f1",
    "span_char_f1",
    "span_relaxed_span_f1",
]

summary = all_metrics[summary_cols].copy()
summary = summary.sort_values(
    ["sentence_macro_f1", "span_char_f1"],
    ascending=False,
)

summary.to_csv("/kaggle/working/results/kaggle_full/final_summary_for_readme.csv", index=False)
display(summary)


In [ ]:
# This archive intentionally excludes model weights and optimizer checkpoints.
!cd /kaggle/working && rm -f full_results_light.zip
!cd /kaggle/working && zip -r full_results_light.zip \
  results/kaggle_full \
  artifacts/kaggle_full/training_result.json \
  artifacts/kaggle_full/threshold_info.json \
  artifacts/kaggle_full/evaluation_summary.json \
  artifacts/kaggle_full/dataset_paths.json


In [ ]:
!ls -lh /kaggle/working/full_results_light.zip
!ls -lh /kaggle/working/results/kaggle_full


In [ ]:
from IPython.display import FileLink, display

display(FileLink("/kaggle/working/full_results_light.zip"))

# If the direct link does not work in the interactive session, save/commit
# the Kaggle notebook version and download the file from the notebook Output tab.


## Final notes

The final full run should produce:
- real LettuceDetect baseline availability;
- full `all_metrics.csv`;
- full `per_type_metrics.csv`;
- full dataset statistics;
- lightweight archive `full_results_light.zip`.

Do not commit model checkpoints or `.safetensors` files to GitHub.
Only commit:
- report files;
- selected CSV result tables;
- README result summary;
- source code and configs.
